In [25]:
import gc
import json
import sqlite3
import uuid
from pathlib import Path
from collections import defaultdict

from qdrant_client import QdrantClient, models
from tuutrag.data import D

In [8]:
DB_PATH: Path = "../../tuutrag.db"

QDRANT_HOST: str      = "localhost"
QDRANT_PORT: int      = 6333
EMBEDDING_DIM: int    = 3072
DISTANCE: models.Distance = models.Distance.COSINE

BRANCH_COLLECTION: str = "branches"
LEAF_COLLECTION: str   = "leafs"

UPSERT_BATCH: int = 256
SQL_FETCH: int = UPSERT_BATCH

In [9]:
qclient = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT, timeout=120, https=False)
print(f"Connected to Qdrant at http://{QDRANT_HOST}:{QDRANT_PORT}/dashboard")

Connected to Qdrant at http://localhost:6333/dashboard


In [10]:
for name in (BRANCH_COLLECTION, LEAF_COLLECTION):
    if qclient.collection_exists(name):
        qclient.delete_collection(name)

    qclient.create_collection(
        collection_name=name,
        vectors_config=models.VectorParams(
            size=EMBEDDING_DIM,
            distance=DISTANCE,
        ),
    )
    print(f"Collection '{name}' created ({EMBEDDING_DIM}-dim, {DISTANCE.name})")

Collection 'branches' created (3072-dim, COSINE)
Collection 'leafs' created (3072-dim, COSINE)


In [11]:
qclient.create_payload_index(BRANCH_COLLECTION, "artifact_uuid",
                             field_schema=models.PayloadSchemaType.KEYWORD)
qclient.create_payload_index(BRANCH_COLLECTION, "type",
                             field_schema=models.PayloadSchemaType.KEYWORD)

qclient.create_payload_index(LEAF_COLLECTION, "branch_uuid",
                             field_schema=models.PayloadSchemaType.KEYWORD)
qclient.create_payload_index(LEAF_COLLECTION, "artifact_uuid",
                             field_schema=models.PayloadSchemaType.KEYWORD)

print("Payload indexes created.")

Payload indexes created.


In [12]:
def qdrant_point_id(text_uuid: str) -> str:
    """Deterministic point ID from the text UUID."""
    return str(uuid.uuid5(uuid.NAMESPACE_DNS, text_uuid))

def iter_rows(cursor: sqlite3.Cursor, size: int = SQL_FETCH):
    """Yield rows in pages from a cursor."""
    while True:
        rows = cursor.fetchmany(size)
        if not rows:
            break
        yield rows

In [13]:
conn = sqlite3.connect(str(DB_PATH))
conn.row_factory = sqlite3.Row

cur = conn.cursor()
cur.execute("""
    SELECT
        be.branch_uuid,
        be.embedding,
        b.artifact_uuid,
        b.chunk,
        b.path,
        a.type
    FROM branch_embeddings  be
    JOIN branches           b  ON b.uuid = be.branch_uuid
    JOIN artifacts          a  ON a.uuid = b.artifact_uuid
""")

branch_count = 0

for page in iter_rows(cur):
    points = []
    for row in page:
        vec = json.loads(row["embedding"])
        points.append(
            models.PointStruct(
                id=qdrant_point_id(row["branch_uuid"]),
                vector=vec,
                payload={
                    "uuid":          row["branch_uuid"],
                    "artifact_uuid": row["artifact_uuid"],
                    "type":          row["type"],
                    "chunk":         row["chunk"],
                    "path":          row["path"],
                },
            )
        )
    qclient.upsert(collection_name=BRANCH_COLLECTION, points=points, wait=True)
    branch_count += len(points)
    del points, page
    gc.collect()

    if branch_count % (UPSERT_BATCH * 10) == 0:
        print(f"  branches upserted: {branch_count:,}")

cur.close()
print(f"Branch upsert complete — {branch_count:,} points")

  branches upserted: 2,560
  branches upserted: 5,120
  branches upserted: 7,680
  branches upserted: 10,240
  branches upserted: 12,800
  branches upserted: 15,360
  branches upserted: 17,920
  branches upserted: 20,480
  branches upserted: 23,040
  branches upserted: 25,600
  branches upserted: 28,160
  branches upserted: 30,720
  branches upserted: 33,280
  branches upserted: 35,840
Branch upsert complete — 37,239 points


In [ ]:
cur = conn.cursor()
cur.execute("""
    SELECT
        le.leaf_uuid,
        le.embedding,
        l.branch_uuid,
        l.text,
        b.artifact_uuid,
        b.path,
        a.type
    FROM leaf_embeddings  le
    JOIN leafs            l  ON l.uuid        = le.leaf_uuid
    JOIN branches         b  ON b.uuid        = l.branch_uuid
    JOIN artifacts        a  ON a.uuid        = b.artifact_uuid
""")

leaf_count = 0

for page in iter_rows(cur):
    points = []
    for row in page:
        vec = json.loads(row["embedding"])
        points.append(
            models.PointStruct(
                id=qdrant_point_id(row["leaf_uuid"]),
                vector=vec,
                payload={
                    "uuid":          row["leaf_uuid"],
                    "branch_uuid":   row["branch_uuid"],
                    "artifact_uuid": row["artifact_uuid"],
                    "type":          row["type"],
                    "text":          row["text"],
                    "path":          row["path"],
                },
            )
        )
    qclient.upsert(collection_name=LEAF_COLLECTION, points=points, wait=True)
    leaf_count += len(points)
    del points, page
    gc.collect()

    if leaf_count % (UPSERT_BATCH * 10) == 0:
        print(f"  leafs upserted: {leaf_count:,}")

cur.close()
conn.close()
print(f"Leaf upsert complete — {leaf_count:,} points")

In [16]:
for name in (BRANCH_COLLECTION, LEAF_COLLECTION):
    info = qclient.get_collection(name)
    exact = qclient.count(collection_name=name, exact=True).count
    print(f"\n{name}:")
    print(f"  points:       {info.points_count:,}")
    print(f"  exact count:  {exact:,}")
    print(f"  index status: {info.status}")


branches:
  points:       37,239
  exact count:  37,239
  index status: green

leafs:
  points:       322,120
  exact count:  322,120
  index status: yellow


In [18]:
def search_similar_branches(
    branch_uuid: str,
    k: int = 10,
    type_filter: str | None = None,
) -> list[models.ScoredPoint]:
    """
    Find top-k branches most similar to `branch_uuid`.

    Parameters
    ----------
    branch_uuid : e.g. "041f16b3-…d300.1"
    k           : number of results
    type_filter : optional — restrict results to a specific artifact type
                  (e.g. "supplemental", "requirement", "test")
    """
    point_id = qdrant_point_id(branch_uuid)

    query_filter = None
    if type_filter:
        query_filter = models.Filter(
            must=[
                models.FieldCondition(
                    key="type",
                    match=models.MatchValue(value=type_filter),
                )
            ]
        )

    return qclient.query_points(
        collection_name=BRANCH_COLLECTION,
        query=point_id,
        using=None,
        limit=k,
        query_filter=query_filter,
        with_payload=True,
    ).points


def search_similar_leafs(
    leaf_uuid: str,
    k: int = 10,
    artifact_uuid_filter: str | None = None,
    branch_uuid_filter: str | None = None,
) -> list[models.ScoredPoint]:
    """
    Find top-k leafs most similar to `leaf_uuid`.

    Parameters
    ----------
    leaf_uuid             : e.g. "041f16b3-…d300.1.1"
    k                     : number of results
    artifact_uuid_filter  : restrict to leafs under a specific artifact
    branch_uuid_filter    : restrict to leafs under a specific branch
    """
    point_id = qdrant_point_id(leaf_uuid)

    conditions = []
    if artifact_uuid_filter:
        conditions.append(
            models.FieldCondition(
                key="artifact_uuid",
                match=models.MatchValue(value=artifact_uuid_filter),
            )
        )
    if branch_uuid_filter:
        conditions.append(
            models.FieldCondition(
                key="branch_uuid",
                match=models.MatchValue(value=branch_uuid_filter),
            )
        )

    query_filter = models.Filter(must=conditions) if conditions else None

    return qclient.query_points(
        collection_name=LEAF_COLLECTION,
        query=point_id,
        using=None,
        limit=k,
        query_filter=query_filter,
        with_payload=True,
    ).points

In [21]:
results = search_similar_branches("041f16b3-f3b7-488f-a27c-074e63a9d300.1", k=3)
for r in results:
    print(f"  {r.score:.4f}  {r.payload['uuid']}  {r.payload['type']}    {r.payload['chunk']}")

  0.9999  58460460-bc83-40cc-ae02-e24d41e050ac.1  supplemental    [{'dataDictionary': {'LDD Version': '1.24.0.0', 'Description': 'This document is a dump of the contents of the PDS4 Data Dictionary', 'IM Version': '1.24.0.0', 'ExternTermMapDictionary': [{'infoModelVersionId': '1.24.0.0', 'datetime': '2025-04-28T22:35:24Z', 'lddVersion': None, 'termmaps': [], 'title': 'PDS4 Term Mappings', 'lddName': None}], 'Title': 'PDS4 Data Dictionary', 'dataTypeDictionary': [{'DataType': {'minimumValue': 'Unbounded', 'identifier': '0001_NASA_PDS_1.pds.ASCII_AnyURI', 'registrationAuthorityId': '0001_NASA_PDS_1', 'minimumCharacters': 'Unbounded', 'maximumCharacters': 'Unbounded', 'nameSpaceId': 'pds', 'title': 'ASCII_AnyURI', 'maximumValue': 'Unbounded'}}, {'DataType': {'minimumValue': 'Unbounded', 'identifier': '0001_NASA_PDS_1.pds.ASCII_BibCode', 'registrationAuthorityId': '0001_NASA_PDS_1', 'minimumCharacters': 'Unbounded', 'maximumCharacters': 'Unbounded', 'nameSpaceId': 'pds', 'title': 'ASCII_Bi

In [22]:
results = search_similar_leafs(
    "041f16b3-f3b7-488f-a27c-074e63a9d300.1.1",
    k=5,
    artifact_uuid_filter="041f16b3-f3b7-488f-a27c-074e63a9d300"
)
for r in results:
    print(f"  {r.score:.4f}  {r.payload['uuid']}  {r.payload['branch_uuid']}")

  0.9214  041f16b3-f3b7-488f-a27c-074e63a9d300.547.6  041f16b3-f3b7-488f-a27c-074e63a9d300.547
  0.9153  041f16b3-f3b7-488f-a27c-074e63a9d300.6.4  041f16b3-f3b7-488f-a27c-074e63a9d300.6
  0.9145  041f16b3-f3b7-488f-a27c-074e63a9d300.191.7  041f16b3-f3b7-488f-a27c-074e63a9d300.191
  0.9141  041f16b3-f3b7-488f-a27c-074e63a9d300.8.2  041f16b3-f3b7-488f-a27c-074e63a9d300.8
  0.9138  041f16b3-f3b7-488f-a27c-074e63a9d300.2.6  041f16b3-f3b7-488f-a27c-074e63a9d300.2


In [23]:
def artifact_uuid_from_branch(branch_uuid: str) -> str:
    """'aaaa-bbbb.3' → 'aaaa-bbbb'"""
    return branch_uuid.rsplit(".", 1)[0]

In [26]:
# Scroll through every point in the branches collection and group by artifact_uuid
artifact_branches: dict[str, list[str]] = defaultdict(list)

offset = None
page_size = 5000
total_scrolled = 0

while True:
    records, next_offset = qclient.scroll(
        collection_name=BRANCH_COLLECTION,
        limit=page_size,
        offset=offset,
        with_payload=["uuid", "artifact_uuid"],
        with_vectors=False,
    )
    for record in records:
        b_uuid = record.payload["uuid"]
        a_uuid = record.payload["artifact_uuid"]
        artifact_branches[a_uuid].append(b_uuid)
    total_scrolled += len(records)

    if next_offset is None:
        break
    offset = next_offset

print(f"Artifacts:       {len(artifact_branches):,}")
print(f"Total branches:  {total_scrolled:,}")

Artifacts:       453
Total branches:  37,239


In [31]:
entity_conn = sqlite3.connect(str(DB_PATH))
entity_conn.row_factory = sqlite3.Row


def get_entities_for_branches(branch_uuids: list[str]) -> list[str]:
    """
    Query the DB for all leafs belonging to the given branch UUIDs,
    parse their JSON entity arrays, and return one combined flat list.
    """
    if not branch_uuids:
        return []

    placeholders = ",".join("?" for _ in branch_uuids)
    cur = entity_conn.cursor()
    cur.execute(
        f"""
        SELECT b.uuid AS branch_uuid, l.entities
        FROM   branches b
        LEFT JOIN leafs l ON b.uuid = l.branch_uuid
        WHERE  b.uuid IN ({placeholders})
        """,
        branch_uuids,
    )

    combined: list[str] = []
    for row in cur:
        raw = row["entities"]
        if raw:
            parsed: list[str] = json.loads(raw)
            combined.extend(parsed)
    cur.close()

    return combined

In [32]:
output: list[dict] = []
processed = 0
total_branches = sum(len(v) for v in artifact_branches.values())

for a_uuid, branches_in_artifact in artifact_branches.items():

    for branch_uuid in branches_in_artifact:
        # Filter at the Qdrant level: only return branches from the SAME artifact
        point_id = qdrant_point_id(branch_uuid)
        results = qclient.query_points(
            collection_name=BRANCH_COLLECTION,
            query=point_id,
            using=None,
            limit=3,
            query_filter=models.Filter(
                must=[
                    models.FieldCondition(
                        key="artifact_uuid",
                        match=models.MatchValue(value=a_uuid),
                    )
                ]
            ),
            with_payload=True,
        ).points

        # Get the current branch's chunk from Qdrant
        current_point = qclient.retrieve(
            collection_name=BRANCH_COLLECTION,
            ids=[qdrant_point_id(branch_uuid)],
            with_payload=True,
            with_vectors=False,
        )
        current_chunk = current_point[0].payload["chunk"] if current_point else ""

        # Collect the current branch + 3 similar branch UUIDs (4 total)
        sim_uuids = [r.payload["uuid"] for r in results]
        all_uuids = [branch_uuid] + sim_uuids

        # Get combined entities for all 4 branches as one flat list
        combined_entities = get_entities_for_branches(all_uuids)

        # Build sim_branches: current branch first, then the 3 similar
        sim_branches = [{"uuid": branch_uuid, "chunk": current_chunk}]
        sim_branches.extend(
            {"uuid": r.payload["uuid"], "chunk": r.payload["chunk"]}
            for r in results
        )

        record = {
            "uuid": branch_uuid,
            "entities": combined_entities,
            "sim_branches": sim_branches,
        }
        output.append(record)

        processed += 1
        if processed % 1_000 == 0:
            print(f"  processed {processed:,} / {total_branches:,} branches")

print(f"\nDone — {processed:,} branch records built, "
      f"with {sum(len(r['sim_branches']) for r in output):,} total similarity entries")

  processed 1,000 / 37,239 branches
  processed 2,000 / 37,239 branches
  processed 3,000 / 37,239 branches
  processed 4,000 / 37,239 branches
  processed 5,000 / 37,239 branches
  processed 6,000 / 37,239 branches
  processed 7,000 / 37,239 branches
  processed 8,000 / 37,239 branches
  processed 9,000 / 37,239 branches
  processed 10,000 / 37,239 branches
  processed 11,000 / 37,239 branches
  processed 12,000 / 37,239 branches
  processed 13,000 / 37,239 branches
  processed 14,000 / 37,239 branches
  processed 15,000 / 37,239 branches
  processed 16,000 / 37,239 branches
  processed 17,000 / 37,239 branches
  processed 18,000 / 37,239 branches
  processed 19,000 / 37,239 branches
  processed 20,000 / 37,239 branches
  processed 21,000 / 37,239 branches
  processed 22,000 / 37,239 branches
  processed 23,000 / 37,239 branches
  processed 24,000 / 37,239 branches
  processed 25,000 / 37,239 branches
  processed 26,000 / 37,239 branches
  processed 27,000 / 37,239 branches
  processe

In [33]:
OUTPUT_FILE = "../data/processed/vector_sim_branches.json"

with open(OUTPUT_FILE, "w") as f:
    json.dump(output, f, indent=2)

import os
file_size = os.path.getsize(OUTPUT_FILE)
print(f"Written to: {os.path.abspath(OUTPUT_FILE)}")
print(f"File size:  {file_size / (1024 * 1024):.1f} MB  ({file_size:,} bytes)")
print(f"Records:    {len(output):,}")

# Close the entity SQLite connection
entity_conn.close()

Written to: /Users/marlonselvi/Documents/Programming/tuutrag-open/data/processed/vector_sim_branches.json
File size:  398.1 MB  (417,457,904 bytes)
Records:    37,239


In [36]:
def append_artifact_summary(data: list[dict], summaries_path: str, output_file: str = "vector_sim_branches.json") -> list[dict]:
    """
    For every record in `data`, look up the artifact summary from
    artifact_summaries.json using branch_uuid → artifact_uuid mapping,
    insert 'artifact_summary' into the record, and write to JSON.
    """
    # Load artifact summaries from JSON
    with open(summaries_path, "r") as f:
        summaries_list: list[dict] = json.load(f)

    # Build artifact_uuid → summary text lookup
    summary_map: dict[str, str] = {
        item["uuid"]: item["text"]
        for item in summaries_list
    }
    print(f"Loaded {len(summary_map):,} artifact summaries from {summaries_path}")

    # Extract artifact_uuid from branch_uuid: "aaaa-bbbb.3" → "aaaa-bbbb"
    def artifact_uuid_from_branch(branch_uuid: str) -> str:
        return branch_uuid.rsplit(".", 1)[0]

    matched = 0
    for record in data:
        a_uuid = artifact_uuid_from_branch(record["uuid"])
        summary = summary_map.get(a_uuid, "")
        record["artifact_summary"] = summary
        if summary:
            matched += 1

    # Reorder keys: uuid → entities → artifact_summary → sim_branches
    reordered: list[dict] = []
    for record in data:
        reordered.append({
            "uuid": record["uuid"],
            "entities": record["entities"],
            "artifact_summary": record["artifact_summary"],
            "sim_branches": record["sim_branches"],
        })

    print(f"Matched summaries:  {matched:,} / {len(reordered):,} records")

    # Write to JSON
    with open(output_file, "w") as f:
        json.dump(reordered, f, indent=2)

    file_size = os.path.getsize(output_file)
    print(f"Written to: {os.path.abspath(output_file)}")
    print(f"File size:  {file_size / (1024 * 1024):.1f} MB  ({file_size:,} bytes)")
    print(f"Records:    {len(reordered):,}")

    return reordered


SUMMARIES_PATH = "../data/processed/artifact_summaries.json"
output = append_artifact_summary(output, SUMMARIES_PATH)

Loaded 453 artifact summaries from ../data/processed/artifact_summaries.json
Matched summaries:  37,239 / 37,239 records
Written to: /Users/marlonselvi/Documents/Programming/tuutrag-open/notebooks/vector_sim_branches.json
File size:  447.9 MB  (469,663,810 bytes)
Records:    37,239


In [37]:
# Check what branch UUIDs look like vs what artifact UUIDs look like
print("Sample branch UUIDs from output:")
for r in output[:5]:
    print(f"  {r['uuid']}")

print("\nSample artifact UUIDs from summaries file:")
with open(SUMMARIES_PATH, "r") as f:
    summaries = json.load(f)
for s in summaries[:5]:
    print(f"  {s['uuid']}")

print("\nDoes rsplit work?")
sample = output[0]["uuid"]
print(f"  branch:   {sample}")
print(f"  rsplit:   {sample.rsplit('.', 1)[0]}")

Sample branch UUIDs from output:
  88f8ed99-f66a-45cd-945a-4ab449b602e9.648
  88f8ed99-f66a-45cd-945a-4ab449b602e9.669
  88f8ed99-f66a-45cd-945a-4ab449b602e9.817
  88f8ed99-f66a-45cd-945a-4ab449b602e9.564
  88f8ed99-f66a-45cd-945a-4ab449b602e9.695

Sample artifact UUIDs from summaries file:
  041f16b3-f3b7-488f-a27c-074e63a9d300
  412d3a83-5c23-4f1a-ad0f-26618b57d266
  39fc3c2d-959e-4d96-b3fd-ee4b9bf81745
  3b3704a7-5302-4980-84d3-7cb479c03d06
  dd987c0f-9f62-4aa5-948b-577848b9f904

Does rsplit work?
  branch:   88f8ed99-f66a-45cd-945a-4ab449b602e9.648
  rsplit:   88f8ed99-f66a-45cd-945a-4ab449b602e9


In [39]:
SUMMARIES_PATH = "../data/processed/artifact_summaries.json"
with open(SUMMARIES_PATH, "r") as f:
    summaries = json.load(f)

summary_uuids = {s["uuid"] for s in summaries}

# Check how many output records map to an artifact that has a summary
matched = 0
unmatched_artifacts = set()
for r in output:
    a_uuid = r["uuid"].rsplit(".", 1)[0]
    if a_uuid in summary_uuids:
        matched += 1
    else:
        unmatched_artifacts.add(a_uuid)

print(f"Total records:           {len(output):,}")
print(f"Matched to a summary:    {matched:,}")
print(f"Unmatched artifacts:     {len(unmatched_artifacts):,}")
print(f"Summaries available for: {len(summary_uuids):,} artifacts")
print(f"\nSample unmatched artifact UUIDs:")
for u in list(unmatched_artifacts)[:10]:
    print(f"  {u}")

Total records:           37,239
Matched to a summary:    37,239
Unmatched artifacts:     0
Summaries available for: 453 artifacts

Sample unmatched artifact UUIDs:


In [ ]:
SUMMARIES_PATH = "../data/processed/artifact_summaries.json"
output = append_artifact_summary(output, SUMMARIES_PATH)

# Verify it's in the file
print("\n--- Verification ---")
with open("vector_sim_branches.json", "r") as f:
    check = json.load(f)

print(f"Keys in first record: {list(check[0].keys())}")
print(f"artifact_summary present: {'artifact_summary' in check[0]}")
print(f"artifact_summary length:  {len(check[0]['artifact_summary'])} chars")
print(f"artifact_summary preview: {check[0]['artifact_summary'][:200]}...")

In [44]:
def append_local_relations(data: list[dict], relations_path: str, output_file: str = "vector_sim_branches.json") -> list[dict]:
    """
    For every record in `data`, look up local relations from
    local_relations_fixed.json by branch_uuid, insert 'local_relations'
    into the record, and write to JSON.
    """
    with open(relations_path, "r") as f:
        relations_list: list[dict] = json.load(f)

    # Build branch_uuid → list of [source, relation, target] triples
    relations_map: dict[str, list[list[str]]] = {
        item["uuid"]: item["relationships"]
        for item in relations_list
    }
    print(f"Loaded relations for {len(relations_map):,} branches from {relations_path}")

    matched = 0
    for record in data:
        relations = relations_map.get(record["uuid"], [])
        record["local_relations"] = relations
        if relations:
            matched += 1

    # Reorder keys: uuid → entities → artifact_summary → local_relations → sim_branches
    reordered: list[dict] = []
    for record in data:
        reordered.append({
            "uuid": record["uuid"],
            "entities": record["entities"],
            "artifact_summary": record["artifact_summary"],
            "local_relations": record["local_relations"],
            "sim_branches": record["sim_branches"],
        })

    print(f"Matched relations:  {matched:,} / {len(reordered):,} records")

    # Write to JSON
    with open(output_file, "w") as f:
        json.dump(reordered, f, indent=2)

    file_size = os.path.getsize(output_file)
    print(f"Written to: {os.path.abspath(output_file)}")
    print(f"File size:  {file_size / (1024 * 1024):.1f} MB  ({file_size:,} bytes)")
    print(f"Records:    {len(reordered):,}")

    return reordered


RELATIONS_PATH = "../data/processed/local_relations_fixed.json"
output = append_local_relations(output, RELATIONS_PATH)

# Verify
print("\n--- Verification ---")
with open("vector_sim_branches.json", "r") as f:
    check = json.load(f)

print(f"Keys in first record: {list(check[0].keys())}")
print(f"local_relations present: {'local_relations' in check[0]}")
print(f"local_relations count:   {len(check[0]['local_relations'])}")
if check[0]['local_relations']:
    print(f"First relation:          {check[0]['local_relations'][0]}")

Loaded relations for 37,225 branches from ../data/processed/local_relations_fixed.json
Matched relations:  37,221 / 37,239 records
Written to: /Users/marlonselvi/Documents/Programming/tuutrag-open/notebooks/vector_sim_branches.json
File size:  508.0 MB  (532,667,419 bytes)
Records:    37,239

--- Verification ---
Keys in first record: ['uuid', 'entities', 'artifact_summary', 'local_relations', 'sim_branches']
local_relations present: True
local_relations count:   15
First relation:          ['Second', 'is a', 'SI Base Unit']


In [47]:
# %% Imports & setup (same as before)
import json
import os
import sqlite3
from collections import defaultdict

# Assumes qclient, BRANCH_COLLECTION, qdrant_point_id, models, DB_PATH
# are already defined in your environment

def artifact_uuid_from_branch(branch_uuid: str) -> str:
    """'aaaa-bbbb.3' → 'aaaa-bbbb'"""
    return branch_uuid.rsplit(".", 1)[0]

# %%
# Scroll through every point in the branches collection and group by artifact_uuid
artifact_branches: dict[str, list[str]] = defaultdict(list)

offset = None
page_size = 5000
total_scrolled = 0

while True:
    records, next_offset = qclient.scroll(
        collection_name=BRANCH_COLLECTION,
        limit=page_size,
        offset=offset,
        with_payload=["uuid", "artifact_uuid"],
        with_vectors=False,
    )
    for record in records:
        b_uuid = record.payload["uuid"]
        a_uuid = record.payload["artifact_uuid"]
        artifact_branches[a_uuid].append(b_uuid)
    total_scrolled += len(records)

    if next_offset is None:
        break
    offset = next_offset

print(f"Artifacts:       {len(artifact_branches):,}")
print(f"Total branches:  {total_scrolled:,}")

# %%
# Load artifact summaries into a lookup map
SUMMARIES_PATH = "../data/processed/artifact_summaries.json"
with open(SUMMARIES_PATH, "r") as f:
    summaries_list: list[dict] = json.load(f)

summary_map: dict[str, str] = {
    item["uuid"]: item["text"]
    for item in summaries_list
}
print(f"Loaded {len(summary_map):,} artifact summaries")

# %%
# Load global relations from vector_sim_branches.json into a lookup map
RELATIONS_PATH = "../data/processed/vector_sim_branches.json"
with open(RELATIONS_PATH, "r") as f:
    relations_list: list[dict] = json.load(f)

# Build branch_uuid → list of [entity, relation, entity] triples
# Parse "Entity1, relation, Entity2" strings into [Entity1, relation, Entity2] lists
relations_map: dict[str, list[list[str]]] = {}
for item in relations_list:
    parsed_rels = []
    for rel_str in item.get("relationships", []):
        parts = [p.strip() for p in rel_str.split(",", 2)]
        if len(parts) == 3:
            parsed_rels.append(parts)
        else:
            parsed_rels.append([rel_str])  # fallback: keep raw if parsing fails
    relations_map[item["uuid"]] = parsed_rels

print(f"Loaded relations for {len(relations_map):,} branches")

# %%
# SQLite entity lookup
entity_conn = sqlite3.connect(str(DB_PATH))
entity_conn.row_factory = sqlite3.Row


def get_entities_for_branches(branch_uuids: list[str]) -> list[str]:
    """
    Query the DB for all leafs belonging to the given branch UUIDs,
    parse their JSON entity arrays, and return one combined flat list.
    """
    if not branch_uuids:
        return []

    placeholders = ",".join("?" for _ in branch_uuids)
    cur = entity_conn.cursor()
    cur.execute(
        f"""
        SELECT b.uuid AS branch_uuid, l.entities
        FROM   branches b
        LEFT JOIN leafs l ON b.uuid = l.branch_uuid
        WHERE  b.uuid IN ({placeholders})
        """,
        branch_uuids,
    )

    combined: list[str] = []
    for row in cur:
        raw = row["entities"]
        if raw:
            parsed: list[str] = json.loads(raw)
            combined.extend(parsed)
    cur.close()

    return combined

# %%
# Main loop: for each branch, find top similar branches from OTHER artifacts
output: list[dict] = []
processed = 0
total_branches = sum(len(v) for v in artifact_branches.values())

for a_uuid, branches_in_artifact in artifact_branches.items():

    for branch_uuid in branches_in_artifact:
        # -----------------------------------------------------------------
        # KEY CHANGE: filter to branches NOT in the same artifact
        # must_not on artifact_uuid == current artifact's UUID
        # -----------------------------------------------------------------
        point_id = qdrant_point_id(branch_uuid)
        results = qclient.query_points(
            collection_name=BRANCH_COLLECTION,
            query=point_id,
            using=None,
            limit=3,
            query_filter=models.Filter(
                must_not=[
                    models.FieldCondition(
                        key="artifact_uuid",
                        match=models.MatchValue(value=a_uuid),
                    )
                ]
            ),
            with_payload=True,
        ).points

        # Collect the similar branch UUIDs (from OTHER artifacts)
        sim_uuids = [r.payload["uuid"] for r in results]

        # All branch UUIDs: current + similar (for entity lookup)
        all_uuids = [branch_uuid] + sim_uuids

        # 1) ENTITIES: combined from current branch + similar branches
        combined_entities = get_entities_for_branches(all_uuids)

        # 2) ARTIFACT SUMMARIES: from the current branch's artifact +
        #    the artifacts of the top similar branches
        artifact_uuids_involved = set()
        artifact_uuids_involved.add(a_uuid)  # current branch's artifact
        for s_uuid in sim_uuids:
            artifact_uuids_involved.add(artifact_uuid_from_branch(s_uuid))

        artifact_summaries = []
        for art_uuid in artifact_uuids_involved:
            summary = summary_map.get(art_uuid, "")
            if summary:
                artifact_summaries.append(summary)

        # 3) GLOBAL RELATIONS: from vector_sim_branches.json for THIS branch
        global_relations = relations_map.get(branch_uuid, [])

        # 4) SIM BRANCHES: list of {uuid, branch_summary} for similar branches
        #    branch_summary comes from the "chunk" payload in Qdrant
        sim_branches = []
        for r in results:
            sim_branches.append({
                "uuid": r.payload["uuid"],
                "branch_summary": r.payload.get("chunk", ""),
            })

        # Build the output record
        record = {
            "uuid": branch_uuid,
            "entities": combined_entities,
            "artifact_summaries": artifact_summaries,
            "global_relations": global_relations,
            "sim_branches": sim_branches,
        }
        output.append(record)

        processed += 1
        if processed % 1_000 == 0:
            print(f"  processed {processed:,} / {total_branches:,} branches")

print(f"\nDone — {processed:,} branch records built, "
      f"with {sum(len(r['sim_branches']) for r in output):,} total similarity entries")

# %%
# Write output to global_vector_sim.json
OUTPUT_FILE = "../data/processed/global_vector_sim.json"

with open(OUTPUT_FILE, "w") as f:
    json.dump(output, f, indent=2)

file_size = os.path.getsize(OUTPUT_FILE)
print(f"Written to: {os.path.abspath(OUTPUT_FILE)}")
print(f"File size:  {file_size / (1024 * 1024):.1f} MB  ({file_size:,} bytes)")
print(f"Records:    {len(output):,}")

# Close the entity SQLite connection
entity_conn.close()

# %%
# Verification
print("\n--- Verification ---")
with open(OUTPUT_FILE, "r") as f:
    check = json.load(f)

print(f"Total records:          {len(check):,}")
print(f"Keys in first record:   {list(check[0].keys())}")
print(f"uuid:                   {check[0]['uuid']}")
print(f"entities count:         {len(check[0]['entities'])}")
print(f"artifact_summaries cnt: {len(check[0]['artifact_summaries'])}")
print(f"global_relations count: {len(check[0]['global_relations'])}")
print(f"sim_branches count:     {len(check[0]['sim_branches'])}")

if check[0]['sim_branches']:
    print(f"\nFirst sim_branch:")
    print(f"  uuid:           {check[0]['sim_branches'][0]['uuid']}")
    print(f"  branch_summary: {check[0]['sim_branches'][0]['branch_summary'][:200]}...")

# Verify cross-artifact: no sim_branch should share the same artifact as the source
cross_violations = 0
for rec in check:
    src_artifact = artifact_uuid_from_branch(rec["uuid"])
    for sb in rec["sim_branches"]:
        sim_artifact = artifact_uuid_from_branch(sb["uuid"])
        if sim_artifact == src_artifact:
            cross_violations += 1

print(f"\nCross-artifact violations: {cross_violations} (should be 0)")

Artifacts:       453
Total branches:  37,239
Loaded 453 artifact summaries
Loaded relations for 37,239 branches
  processed 1,000 / 37,239 branches
  processed 2,000 / 37,239 branches
  processed 3,000 / 37,239 branches
  processed 4,000 / 37,239 branches
  processed 5,000 / 37,239 branches
  processed 6,000 / 37,239 branches
  processed 7,000 / 37,239 branches
  processed 8,000 / 37,239 branches
  processed 9,000 / 37,239 branches
  processed 10,000 / 37,239 branches
  processed 11,000 / 37,239 branches
  processed 12,000 / 37,239 branches
  processed 13,000 / 37,239 branches
  processed 14,000 / 37,239 branches
  processed 15,000 / 37,239 branches
  processed 16,000 / 37,239 branches
  processed 17,000 / 37,239 branches
  processed 18,000 / 37,239 branches
  processed 19,000 / 37,239 branches
  processed 20,000 / 37,239 branches
  processed 21,000 / 37,239 branches
  processed 22,000 / 37,239 branches
  processed 23,000 / 37,239 branches
  processed 24,000 / 37,239 branches
  process

In [48]:
# %%
# Patch global_relations into the already-built output (no need to re-query Qdrant)
patched = 0
for record in output:
    rels = relations_map.get(record["uuid"], [])
    record["global_relations"] = rels
    if rels:
        patched += 1

print(f"Patched global_relations: {patched:,} / {len(output):,} records")

# Rewrite the file
OUTPUT_FILE = "../data/processed/global_vector_sim.json"
with open(OUTPUT_FILE, "w") as f:
    json.dump(output, f, indent=2)

file_size = os.path.getsize(OUTPUT_FILE)
print(f"Written to: {os.path.abspath(OUTPUT_FILE)}")
print(f"File size:  {file_size / (1024 * 1024):.1f} MB  ({file_size:,} bytes)")

# Quick verification
print(f"\nFirst record global_relations count: {len(output[0]['global_relations'])}")
if output[0]['global_relations']:
    print(f"First relation: {output[0]['global_relations'][0]}")

Patched global_relations: 0 / 37,239 records
Written to: /Users/marlonselvi/Documents/Programming/tuutrag-open/data/processed/global_vector_sim.json
File size:  515.4 MB  (540,482,589 bytes)

First record global_relations count: 0


In [49]:
# %%
# Debug: compare UUIDs between output and relations_map
output_uuids = {r["uuid"] for r in output}
relations_uuids = set(relations_map.keys())

print(f"UUIDs in output:         {len(output_uuids):,}")
print(f"UUIDs in relations_map:  {len(relations_uuids):,}")
print(f"Intersection:            {len(output_uuids & relations_uuids):,}")
print(f"In output but not rels:  {len(output_uuids - relations_uuids):,}")
print(f"In rels but not output:  {len(relations_uuids - output_uuids):,}")

# Show samples from each side
print("\nSample UUIDs from output (first 5):")
for u in list(output_uuids)[:5]:
    print(f"  '{u}'")

print("\nSample UUIDs from relations_map (first 5):")
for u in list(relations_uuids)[:5]:
    print(f"  '{u}'")

# Check for subtle differences (whitespace, encoding, etc.)
sample_out = next(iter(output_uuids))
sample_rel = next(iter(relations_uuids))
print(f"\nOutput sample repr:    {repr(sample_out)}")
print(f"Relations sample repr: {repr(sample_rel)}")
print(f"Output sample bytes:   {sample_out.encode()}")
print(f"Relations sample bytes:{sample_rel.encode()}")


UUIDs in output:         37,239
UUIDs in relations_map:  37,239
Intersection:            37,239
In output but not rels:  0
In rels but not output:  0

Sample UUIDs from output (first 5):
  'f0036c9c-1811-4f62-8e9f-e9e22be81146.325'
  'f0036c9c-1811-4f62-8e9f-e9e22be81146.1009'
  'a9c21a72-981f-44b3-92bc-50bacae2695c.151'
  'dfd11609-01fa-43f2-a50f-0fc5e16c6440.952'
  '412d3a83-5c23-4f1a-ad0f-26618b57d266.860'

Sample UUIDs from relations_map (first 5):
  'f0036c9c-1811-4f62-8e9f-e9e22be81146.325'
  'f0036c9c-1811-4f62-8e9f-e9e22be81146.1009'
  'a9c21a72-981f-44b3-92bc-50bacae2695c.151'
  'dfd11609-01fa-43f2-a50f-0fc5e16c6440.952'
  '412d3a83-5c23-4f1a-ad0f-26618b57d266.860'

Output sample repr:    'f0036c9c-1811-4f62-8e9f-e9e22be81146.325'
Relations sample repr: 'f0036c9c-1811-4f62-8e9f-e9e22be81146.325'
Output sample bytes:   b'f0036c9c-1811-4f62-8e9f-e9e22be81146.325'
Relations sample bytes:b'f0036c9c-1811-4f62-8e9f-e9e22be81146.325'


In [50]:
# %%
# Debug: check what's actually in the relations_map values
has_rels = sum(1 for v in relations_map.values() if v)
empty_rels = sum(1 for v in relations_map.values() if not v)
print(f"relations_map entries with data:  {has_rels:,}")
print(f"relations_map entries empty:      {empty_rels:,}")

# Check what keys exist in the source file
sample_item = relations_list[0]
print(f"\nKeys in first record of vector_sim_branches.json: {list(sample_item.keys())}")
print(f"uuid: {sample_item['uuid']}")

# Check all keys that look like "relation" across the file
all_keys = set()
for item in relations_list:
    all_keys.update(item.keys())
print(f"\nAll unique keys in vector_sim_branches.json: {all_keys}")

# Check if local_relations exists and what it contains
if "local_relations" in sample_item:
    print(f"\nlocal_relations type:  {type(sample_item['local_relations'])}")
    print(f"local_relations count: {len(sample_item['local_relations'])}")
    print(f"local_relations value: {sample_item['local_relations']}")
else:
    print("\n'local_relations' key NOT FOUND in first record")
    # Try to find the right key
    for k, v in sample_item.items():
        if isinstance(v, list) and v and isinstance(v[0], (list, str)):
            print(f"  Candidate key: '{k}' -> type={type(v[0])}, sample={v[0]}")
            

relations_map entries with data:  0
relations_map entries empty:      37,239

Keys in first record of vector_sim_branches.json: ['uuid', 'entities', 'artifact_summary', 'local_relations', 'sim_branches']
uuid: 88f8ed99-f66a-45cd-945a-4ab449b602e9.648

All unique keys in vector_sim_branches.json: {'artifact_summary', 'local_relations', 'sim_branches', 'uuid', 'entities'}

local_relations type:  <class 'list'>
local_relations count: 15
local_relations value: [['Second', 'is a', 'SI Base Unit'], ['Year', 'is a', 'unit of time'], ['Maximum_Value', 'has unit', 'Meter Per Second'], ['Minimum_Value', 'has unit', 'Meter Per Second'], ['Permissible_Value', 'has unit', 'Meter Per Second'], ['Specified_Unit_Id', 'provides unit for', 'Maximum_Value'], ['Specified_Unit_Id', 'provides unit for', 'Minimum_Value'], ['Specified_Unit_Id', 'provides unit for', 'Permissible_Value'], ['Specified_Unit_Id', 'is attribute in', 'Units Of Velocity'], ['Specified_Unit_Id', 'data type is', 'UTF8_Short_String_Coll

In [51]:
# %%
# Rebuild relations_map from the CURRENT relations_list in memory and patch output
relations_map = {
    item["uuid"]: item.get("local_relations", [])
    for item in relations_list
}

has_rels = sum(1 for v in relations_map.values() if v)
print(f"Rebuilt relations_map: {has_rels:,} / {len(relations_map):,} have data")

# Patch output
patched = 0
for record in output:
    rels = relations_map.get(record["uuid"], [])
    record["global_relations"] = rels
    if rels:
        patched += 1

print(f"Patched global_relations: {patched:,} / {len(output):,} records")

# Rewrite the file
OUTPUT_FILE = "../data/processed/global_vector_sim.json"
with open(OUTPUT_FILE, "w") as f:
    json.dump(output, f, indent=2)

file_size = os.path.getsize(OUTPUT_FILE)
print(f"Written to: {os.path.abspath(OUTPUT_FILE)}")
print(f"File size:  {file_size / (1024 * 1024):.1f} MB  ({file_size:,} bytes)")

# Quick verification
print(f"\nFirst record global_relations count: {len(output[0]['global_relations'])}")
if output[0]['global_relations']:
    print(f"First relation: {output[0]['global_relations'][0]}")

Rebuilt relations_map: 37,221 / 37,239 have data
Patched global_relations: 37,221 / 37,239 records
Written to: /Users/marlonselvi/Documents/Programming/tuutrag-open/data/processed/global_vector_sim.json
File size:  574.6 MB  (602,480,745 bytes)

First record global_relations count: 15
First relation: ['Second', 'is a', 'SI Base Unit']
